In [1]:
# LIB IMPORT
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import sqlite3
from sqlalchemy import create_engine
from datetime import datetime, timedelta

In [3]:
# DATA ACQUISITION & PROFILING
## import data from local machine
df = pd.read_csv('dataset_01.csv', encoding = 'latin-1') 

## display data overview
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [ ]:


## display data shape and information
df.info()

## display descriptive stat:
df.describe(include = 'all')

In [97]:
# DATA CLEANING, STRUCTURING & TRANSFORMATION
## check missing values
df.isnull().sum()

## remove missing values
df = df.dropna()

## check duplicates
df.duplicated().sum()

## rename columns
df.columns = ['invoice_no', 'product_id', 'product', 'qty', 'invoice_date', 'price', 'customer_id', 'location']

## format datatypes
df['customer_id'] = df['customer_id'].astype(str)
df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors = 'coerce')

## remove whitespace
df['product'].str.strip()

## validate data
df = df[(df['qty'] > 0) & (df['price'] > 0)]

## create total amount column
df['total_amt'] = df['qty'] * df['price']
## create column of invoice date with year & month format 
df['invoice_year_month'] = df['invoice_date'].dt.to_period('M')
df['invoice_year_month'] = df['invoice_year_month'].astype(str)
## create invoice month column (only in 2011)
df['invoice_month_name_2011'] = pd.to_datetime(np.where(df['invoice_date'].dt.year == 2011, df['invoice_date'], pd.NaT))
df['invoice_month_name_2011'] = df['invoice_month_name_2011'].dt.month_name().str[:3]

## order dataframe
df.sort_values(by = 'invoice_date', inplace = True)

In [ ]:
# DATA ANALYSIS_EDA
## check correlation between price and qty
### observe with pearson correlation coefficient
pearson_coeff = df['price'].corr(df['qty'])
pearson_coeff

### display distribution
plt.scatter (df['price'], df['qty'])
plt.title ('correlation of price vs qty')
plt.xlabel ('price')
plt.ylabel ('qty')
plt.show()

## top 10 location/sales
### aggregate data
location_sales = df.groupby('location')['total_amt'].sum().sort_values(ascending = False).head(10)
location_sales

### display distribution
plt.figure(figsize = (10,3))
sns.barplot(x = location_sales.index, y = location_sales.values)
plt.title('Top 10 Locations per Sales')
plt.xticks(rotation = 90)
plt.xlabel('Location')
plt.ylabel('Sales Amt')
plt.show()

## top 10 customers mostly spent
### aggregate data
customer_mostly_spent = df.groupby('customer_id')['total_amt'].sum().sort_values(ascending = False).head(10)
customer_mostly_spent

### display distribution
plt.figure(figsize = (10,3))
sns.barplot(x = customer_mostly_spent.index, y = customer_mostly_spent.values)
plt.title('Top 10 Customers Mostly Spent')
plt.xlabel('Customer ID')
plt.ylabel('Spent Amt')
plt.xticks(rotation = 'vertical')
plt.show()

## top 10 selling products
### aggregate data
product_mostly_sold_out = df.groupby('product')['total_amt'].sum().sort_values(ascending = False).head(10)
product_mostly_sold_out

### display distribution 
plt.figure(figsize = (10,5))
sns.barplot(x = product_mostly_sold_out.values, y = product_mostly_sold_out.index)
plt.title('Top 10 Products Mostly Soldout')
plt.xlabel('Total Amt')
plt.ylabel('Product')
plt.yticks(rotation = 45)
plt.show()

## sales trend over month
### aggregate data
sales_trend = df.groupby('invoice_year_month')['total_amt'].sum()
sales_trend
### display distribution
plt.figure(figsize = (13,5))
sales_trend.plot()
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Sales Amt')
plt.show()

## top 5 months/sales
### aggregate data
month_sales = df.groupby('invoice_month_2011')['total_amt'].sum().sort_values(ascending = False).head()
month_sales
### display distribution
plt.figure(figsize = (10,3))
sns.barplot(x = month_sales.index, y = month_sales.values)
plt.title('Top 5 Month per Sales')
plt.xlabel('Month')
plt.ylabel('Sales Amt')
plt.show()

In [ ]:
# DATA ANALYSIS_RFM
## define reference_date
snapshot_date = df['invoice_date'].max() + pd.Timedelta(days = 1)
snapshot_date

## aggregate rfm measures
rfm = df.groupby('customer_id').agg(
    {
        'invoice_date' : lambda x : (snapshot_date - x.max()).days,
        'qty' : 'nunique',
        'price' : 'sum'
    }
).reset_index()
rfm.head()

## rename columns
rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']
rfm.head()

## aggregate rfm scores
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels = [5,4,3,2,1])
rfm['f_score'] = pd.qcut(rfm['frequency'], 5, labels = [1,2,3,4,5])
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels = [1,2,3,4,5])
rfm.head()

## format datatypes
rfm['r_score'] = pd.to_numeric(rfm['r_score'])
rfm['f_score'] = pd.to_numeric(rfm['f_score'])
rfm['m_score'] = pd.to_numeric(rfm['m_score'])

## sum rfm scores
rfm['rfm_sum'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

## concatenation of 'r','f','m' scores into 'rfm' scores
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

## segment customer over rfm scores
def rfm_segment(rfm):
    if rfm['rfm_sum'] >= 12:
        return 'Champions'
    elif rfm['rfm_sum'] >= 9:
        return 'Loyal Customers'
    elif rfm['rfm_sum'] >= 6:
        return 'Potential Loyalists'
    elif rfm['rfm_sum'] >= 3:
        return 'Risk Customers'
    else:
        return 'Others'

## create segment column
rfm['segment'] = rfm.apply(rfm_segment, axis = 1)

## customer segmentation comparison
### aggregate data
segment_compare = rfm.groupby('segment')['customer_id'].count().sort_values(ascending = False)
segment_compare

### display distribution
plt.figure(figsize = (10,5))
sns.barplot(x = segment_compare.values, y = segment_compare.index)
plt.title('Customer Segmentation Comparison')
plt.xlabel('No. of Customers')
plt.ylabel('Segmentation')
plt.show()

In [ ]:
#DATA WAREHOUSEING
## create database
engine = create_engine ('sqlite:///cleaned_data_01.db', echo = False)

## store cleaned & transformed data into database
df.to_sql('transactions', con = engine, if_exists = 'replace', index = False)
rfm.to_sql('rfm_analysis', con = engine, if_exists = 'replace', index = False)

## querying from database
pd.read_sql('select segment, count(*) as customers from rfm_analysis group by segment', con = engine)